# TV4 — Bảng chính: train CLIP trên EuroSAT và Aircraft-20

Dựng từ [`template_colab.ipynb`](template_colab.ipynb) bản `master` (ô 3–10 giữ gần nguyên), thêm
ô 11–15 để chạy **cả loạt** của bảng chính rồi ghi `clf_main.csv`.

**Các lần chạy, mỗi dataset 7 lần (cộng 4 lần IsSynth khi có bộ ảnh)**

| | cờ |
|---|---|
| chỉ ảnh thật | `--is_synth_train=False --n_shot=16` |
| chỉ ảnh sinh | `--is_synth_train=True --sd_version=sd2.1 --n_img_per_cls=200` + SD gốc `--n_shot=0` / dset `--n_shot=16 --is_dataset_wise=True` / cls `--n_shot=16 --is_dataset_wise=False` / IsSynth `--n_shot=16 --issynth_strength=<s>` |
| thật + sinh | như trên + `--is_pooled_fewshot=True --lambda_1=0.8` |

**Cách dùng**

1. Secrets: `KAGGLE_USERNAME`, `KAGGLE_KEY` (bật *Notebook access*). *Runtime > Change runtime type* → T4 GPU.
2. **Ô 11 (hyperparameter): TV4 chốt trước khi chạy thật.** Giá trị đang để là đề xuất, xem chú thích trong ô.
3. *Runtime > Run all*. Ô 10 phải ra `KẾT QUẢ: ĐẠT`. Ô 14 chạy lần lượt các lần chạy, **bỏ qua lần đã có
   trong `clf_main.csv` trên Drive**: Colab ngắt giữa chừng thì mở lại và *Run all* lần nữa, nó chạy tiếp.
4. Xong thì ô 15 in bảng so với Table 1. Tải `clf_main.csv` từ Drive về, chép vào `results/clf_main.csv`
   trên máy mình, commit lên nhánh của bạn.

**Lưu ý khi đọc kết quả**

- `acc` = top-1 trên tập test ở **epoch cuối**. `main.py` lưu `best_checkpoint.pth` chọn theo *tập test*;
  số "best" đó lạc quan, nên chỉ ghi vào cột `note` để tham khảo, không đưa vào bảng.
- `main.py` bắt mọi exception rồi chỉ `print` ra `log.log`, nên exit code luôn là 0. Ô 13 xác định lần
  chạy hỏng bằng cách đọc log: thiếu dòng test của epoch nào, hoặc có `Traceback`, là hỏng và không ghi CSV.
- Sau khi gọi `set_log`, `main.py` in **thẳng vào `log.log`**, không ra màn hình. Ô 13 cứ 30 giây
  in lại các dòng test mới.
- Checkpoint (~0,6 GB mỗi lần chạy) bị xoá sau khi đọc xong (`KEEP_CKPT=False`); Drive chỉ giữ log.

In [ ]:
# ===== 2. Tham so - sua o day =====
BRANCH = "hnguyen"  # nhanh co notebook nay + scripts/make_table_main.py; code classify giong master
WORK = "/content"   # o dia cua may Colab: nhanh, mat khi het phien
DRIVE_OUT = "/content/drive/MyDrive/DataDream/outputs/tv4"  # log + clf_main.csv (ben lau)
TEAM_DRIVE = "/content/drive/MyDrive/DataDream"              # ban du phong (vd split_zhou_EuroSAT.json)

FETCH_KAGGLE = True         # anh sinh + tap few-shot da ban giao tren Kaggle (~7,9 GB, tai ca dataset)
FETCH_EUROSAT = True        # EuroSAT (~90 MB) + file chia CoOp
FETCH_AIRCRAFT = True       # FGVC-Aircraft (~2,7 GB) - tap test Aircraft-20
CACHE_AIRCRAFT_TAR = False  # True: giu file tar.gz cua Aircraft tren Drive, lan sau chi giai nen

DATA = f"{WORK}/data"       # anh that
SYN = f"{WORK}/synthetic"   # anh sinh (symlink vao ban Kaggle) = synth_train_data_dir
OUT = f"{WORK}/outputs"     # output cua classify (moi lan chay mot thu muc con)

# --- rieng TV4 ---
DATASETS = ["eurosat", "fgvc_aircraft"]  # thu tu chay: EuroSAT nhe hon, chay truoc
ISSYNTH_STRENGTH = None     # vd 0.3 khi bo IsSynth da len Kaggle (phai khop ten thu muc ..._issynth_str0.3)
ONLY = None                 # None = chay tat ca; hoac list run_name / tien to, vd ["eurosat_real", "aircraft20_dset"]
KEEP_CKPT = False           # True: giu best_checkpoint.pth (~0,6 GB/lan chay) trong /content (khong chep len Drive)
RESULTS_CSV = f"{DRIVE_OUT}/clf_main.csv"  # nguon su that khi dang chay; cuoi cung chep vao results/clf_main.csv

In [ ]:
# ===== 3. O dau tien cua moi notebook (CONTRIBUTING.md muc 6, giu nguyen) =====
from google.colab import drive
drive.mount('/content/drive')

import os
REPO = '/content/DataDream'
if os.path.exists(REPO):
    !cd {REPO} && git pull
else:
    !git clone https://github.com/Vietpn1909/DataDream.git {REPO}
%cd {REPO}

# them: chuyen sang nhanh BRANCH va in commit dang chay (ghi vao cot `commit` cua results/)
!git checkout -q {BRANCH} && git pull -q
COMMIT = !git rev-parse --short=7 HEAD
print("nhanh", BRANCH, "| commit", COMMIT[0])

In [ ]:
# ===== 4. Kiem GPU =====
import sys
import torch
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader || true
print("python", sys.version.split()[0], "| torch", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CANH BAO: KHONG CO GPU. Runtime > Change runtime type > T4 GPU, roi chay lai tu dau.")
    print("(Bai kiem o o 10 van chay duoc tren CPU nhung cham; train CLIP thi phai co GPU.)")

In [ ]:
# ===== 5. Thu vien buoc 3 - dung nhu docs/setup.md "Buoc 3: Colab" =====
# KHONG cai lai torch/torchvision: Colab co san, cai lai se pha moi truong (~10 phut).
# requirements-clf.txt ghim numpy==1.26.4 va Pillow==10.3.0, co the doi ban cua Colab. Neu pip
# doi mot trong hai, o nay DUNG "Run all" lai (kernel van giu ban cu da nap): Runtime > Restart
# session, roi Runtime > Run all lan nua - lan sau pip khong doi gi, o nay chay thang qua.
# Neu phai doi mot phien ban da ghim: sua thang requirements-clf.txt
# trong mot PR [clf] (header cua file do yeu cau vay), dung chi sua trong notebook.
import importlib.metadata as md
before = {k: md.version(k) for k in ("numpy", "Pillow")}
!pip install -q -r requirements-clf.txt
!pip install -q git+https://github.com/openai/CLIP.git
after = {k: md.version(k) for k in ("numpy", "Pillow")}
changed = {k: (before[k], after[k]) for k in before if before[k] != after[k]}
if changed:
    raise RuntimeError("pip da doi %s -> Runtime > Restart session, roi Runtime > Run all lan nua." % changed)
print("numpy", after["numpy"], "| Pillow", after["Pillow"], "- khong can restart")

In [ ]:
# ===== 6. Secrets -> bien moi truong (KHONG BAO GIO in gia tri) =====
import os
from google.colab import userdata
for name, required in (("KAGGLE_USERNAME", True), ("KAGGLE_KEY", True), ("WANDB_API_KEY", False)):
    try:
        os.environ[name] = userdata.get(name)
        print("%-16s da nap" % name)
    except Exception:  # chua tao secret, hoac chua bat Notebook access
        print("%-16s %s" % (name, "THIEU: them o Secrets (bieu tuong chia khoa), bat Notebook access"
                            if required else "khong co (khong bat buoc)"))

In [ ]:
# ===== 7. Anh sinh (va tap few-shot) da ban giao tren Kaggle =====
# NGUON: docs/data.md muc "Dung tren Colab". Bang map duoi day phai GIONG HET ban o do;
# doi cach dat ten thi sua docs/data.md truoc, roi chep lai vao day.
KAGGLE_ROOT = None
if FETCH_KAGGLE:
    try:
        import kagglehub
    except ImportError:
        !pip install -q kagglehub
        import kagglehub
    import os
    root = kagglehub.dataset_download("vitphmnguyn/datadream") + "/handoff"
    MAP = {
        "eurosat_sdbase_K0_n200":    "eurosat/sd2.1/gs2.0_nis50/shot0_template1",
        "eurosat_dset_K16_n200":     "eurosat/sd2.1/gs2.0_nis50/shot16_seed0_template1_lr0.0001_ep200_dswise",
        "eurosat_cls_K16_n200":      "eurosat/sd2.1/gs2.0_nis50/shot16_seed0_template1_lr0.0001_ep200",
        "aircraft20_sdbase_K0_n200": "fgvc_aircraft/sd2.1/gs2.0_nis50/shot0_template1",
        "aircraft20_dset_K16_n200":  "fgvc_aircraft/sd2.1/gs2.0_nis50/shot16_seed0_template1_lr0.0001_ep200_dswise",
        "aircraft20_cls_K16_n200":   "fgvc_aircraft/sd2.1/gs2.0_nis50/shot16_seed0_template1_lr0.0001_ep200",
    }
    if ISSYNTH_STRENGTH is not None:  # TV4 THEM: bo IsSynth (ten zip theo docs/data.md)
        for p, ds in (("eurosat", "eurosat"), ("aircraft20", "fgvc_aircraft")):
            MAP[f"{p}_issynth_K16_n200"] = (f"{ds}/sd2.1/gs2.0_nis50/"
                f"shot16_seed0_template1_issynth_str{float(ISSYNTH_STRENGTH)}")
    for src, dst in MAP.items():
        if not os.path.isdir(f"{root}/synthetic/{src}"):
            print("CHUA CO tren Kaggle:", src)
            continue
        os.makedirs(os.path.dirname(f"{SYN}/{dst}"), exist_ok=True)
        if not os.path.exists(f"{SYN}/{dst}"):
            os.symlink(f"{root}/synthetic/{src}", f"{SYN}/{dst}")
    KAGGLE_ROOT = root
    print("Kaggle:", root)
    !ls {root}/synthetic {root}/real

In [ ]:
# ===== 8. Anh that -> /content/data (o dia may Colab, KHONG phai Drive) =====
import json, os, shutil
import torchvision as tv
os.makedirs(DATA, exist_ok=True)

if FETCH_EUROSAT:
    eu = f"{DATA}/eurosat/train"
    tv.datasets.EuroSAT(root=eu, download=True)  # -> eu/eurosat/2750/<lop>/*.jpg
    split = f"{eu}/split_zhou_EuroSAT.json"       # file chia cua CoOp: classify cham diem tren phan 'test'

    def split_ok(p):
        try:
            with open(p) as f:
                return len(json.load(f)["test"]) == 8100
        except Exception:
            return False

    if not split_ok(split):
        import gdown  # link trong DATASETS.md cua CoOp
        try:
            gdown.download(id="1Ip7yaCWFi0eaOFUGga0lUdVi_DDQth1o", output=split, quiet=True)
        except Exception as e:
            print("gdown loi:", e)
    if not split_ok(split) and os.path.exists(f"{TEAM_DRIVE}/split_zhou_EuroSAT.json"):
        shutil.copy(f"{TEAM_DRIVE}/split_zhou_EuroSAT.json", split)  # ban du phong tren Drive nhom
    print("EuroSAT:", "ok" if split_ok(split) else "THIEU split_zhou_EuroSAT.json -> chep vao " + split)

if FETCH_AIRCRAFT:
    ac = f"{DATA}/fgvc_aircraft/train"
    tar = "fgvc-aircraft-2013b.tar.gz"
    cache = f"{TEAM_DRIVE}/cache/{tar}"
    os.makedirs(ac, exist_ok=True)
    if CACHE_AIRCRAFT_TAR and os.path.exists(cache) and not os.path.isdir(f"{ac}/fgvc-aircraft-2013b"):
        shutil.copy(cache, f"{ac}/{tar}")  # torchvision thay file tar co san -> chi giai nen, khong tai
    # dung loi goi va thu muc ma classify/data.py dung
    tv.datasets.FGVCAircraft(root=ac, split="trainval", annotation_level="variant", download=True)
    if CACHE_AIRCRAFT_TAR and not os.path.exists(cache) and os.path.exists(f"{ac}/{tar}"):
        os.makedirs(os.path.dirname(cache), exist_ok=True)
        shutil.copy(f"{ac}/{tar}", cache)
    print("FGVC-Aircraft: ok")

# Tap few-shot: symlink vao ban Kaggle, de cau truc giong may Viet (<data>/<ds>/real_train_fewshot/seed0)
for ds, name in (("eurosat", "eurosat_fewshot_seed0_K16"), ("fgvc_aircraft", "aircraft20_fewshot_seed0_K16")):
    link = f"{DATA}/{ds}/real_train_fewshot"
    src = f"{KAGGLE_ROOT}/real/{name}" if KAGGLE_ROOT else None
    if src and os.path.isdir(f"{src}/seed0") and not os.path.exists(link):
        os.makedirs(os.path.dirname(link), exist_ok=True)
        os.symlink(src, link)
    print("few-shot %-14s %s" % (ds, link if os.path.isdir(f"{link}/seed0")
                                 else "CHUA CO (%s chua co tren Kaggle, hoac FETCH_KAGGLE=False)" % name))

In [ ]:
# ===== 9. classify/local.yaml (nam trong .gitignore, khong bao gio commit) =====
# classify/config.py doc CA 7 key (configs/local.example.yaml khoi [3]); thieu mot key la KeyError.
import os, yaml
local = {
    # main.py --log=wandb dung key nay. Lay tu Secrets, chi ghi vao file tren may Colab, khong in ra.
    "wandb_key": os.environ.get("WANDB_API_KEY") or "UNUSED_IF_LOG_IS_TENSORBOARD",
    "metadata_dir": "metadata/imagenet",
    "clip_download_dir": f"{WORK}/models/clip",
    "synth_train_data_dir": SYN,
    "real_train_data_dir": {"eurosat": f"{DATA}/eurosat", "fgvc_aircraft": f"{DATA}/fgvc_aircraft"},
    "real_test_data_dir": {"eurosat": f"{DATA}/eurosat", "fgvc_aircraft": f"{DATA}/fgvc_aircraft"},
    # code tu noi them "<fewshot_seed>/" (vd seed0) vao sau
    "real_train_fewshot_data_dir": {"eurosat": f"{DATA}/eurosat/real_train_fewshot",
                                    "fgvc_aircraft": f"{DATA}/fgvc_aircraft/real_train_fewshot"},
}
with open(f"{REPO}/classify/local.yaml", "w") as f:
    yaml.safe_dump(local, f, sort_keys=False)
print(yaml.safe_dump({k: v for k, v in local.items() if k != "wandb_key"}, sort_keys=False))

In [ ]:
# ===== 10. Bai kiem nhanh (vai phut, khong can anh sinh) =====
# (1) tap test / tap that ma classify nap dung chua; (2) CLIP zero-shot EuroSAT tren dung tap
# test do. Ky vong: "KET QUA: DAT", va EuroSAT 8100 anh, top-1 khoang 40% (prompt classify) /
# 48% (prompt reference). Khong truyen --csv o day: chi de kiem moi truong.
%cd {REPO}
!python scripts/test_eval_splits.py --data {DATA}
!python scripts/zeroshot_clip.py --dataset eurosat

In [ ]:
# ===== 11. Hyperparameter - TV4 CHOT TRUOC KHI CHAY THAT =====
# DE XUAT, chua chot. Nguon:
#  - paper (Appendix B): AdamW + cosine, batch 64, lr tim trong {1e-4,1e-5,1e-6,1e-7}, wd trong
#    {5e-4,1e-4}, Mixup/CutMix la mot lua chon khi tim; KHONG cong bo so epoch cua classifier.
#  - classify/bash_run.sh cua repo goc: lr 1e-4, wd 1e-4, min_lr 1e-8, 10 epoch, warmup 3, mix_aug False.
# KHONG de mac dinh cua config.py (lr=0.1, 100 epoch, warmup 25): lr 0.1 pha CLIP + LoRA.
#
# Chi anh that chi co 160 (EuroSAT) / 320 (Aircraft) anh -> 3 / 5 buoc moi epoch voi batch 64. 10 epoch
# = 30 / 50 buoc, qua it. De so buoc gan bang chi-anh-sinh (EuroSAT ~320, Aircraft ~630), de xuat 100
# epoch, warmup 10 cho rieng setting nay. Neu TV4 muon cung so epoch cho moi setting thi xoa HP_REAL_ONLY.
HP = {
    "eurosat":       dict(lr=1e-4, wd=1e-4, min_lr=1e-8, epochs=10, warmup_epochs=3, batch_size=64, is_mix_aug=False),
    "fgvc_aircraft": dict(lr=1e-4, wd=1e-4, min_lr=1e-8, epochs=10, warmup_epochs=3, batch_size=64, is_mix_aug=False),
}
HP_REAL_ONLY = dict(epochs=100, warmup_epochs=10)  # ghi de HP[ds] khi chi train anh that

# batch_size_eval=4 chia het ca 8100 (EuroSAT) lan 668 (Aircraft-20) anh test: top1 cua main.py la
# trung binh theo batch, nen batch cuoi le se lam lech so (rat nho, nhung khong can).
EVAL_BS = 4

for ds in DATASETS:
    print(ds, HP[ds])
print("chi anh that: ghi de", HP_REAL_ONLY)

In [ ]:
# ===== 12. Danh sach lan chay + kiem du lieu truoc khi chay =====
import os

PREFIX = {"eurosat": "eurosat", "fgvc_aircraft": "aircraft20"}
N_CLASSES = {"eurosat": 10, "fgvc_aircraft": 20}
N_IMG = 200
SYN_SUB = {  # thu muc ma classify/config.py tao ra (set_synth_train_data_dir), bo "/train"
    "sdbase": "shot0_template1",
    "dset": "shot16_seed0_template1_lr0.0001_ep200_dswise",
    "cls": "shot16_seed0_template1_lr0.0001_ep200",
}
if ISSYNTH_STRENGTH is not None:
    SYN_SUB["issynth"] = f"shot16_seed0_template1_issynth_str{float(ISSYNTH_STRENGTH)}"


def make_runs():
    runs = []
    for ds in DATASETS:
        p = PREFIX[ds]
        runs.append(dict(run_name=f"{p}_real_K16", dataset=ds, variant="real", K=16, n_img=0, pooled=False))
        for pooled in (False, True):
            for v in SYN_SUB:
                K = 0 if v == "sdbase" else 16
                runs.append(dict(run_name=f"{p}_{v}_K{K}_n{N_IMG}" + ("_pooled" if pooled else ""),
                                 dataset=ds, variant=v, K=K, n_img=N_IMG, pooled=pooled))
    if ONLY:
        runs = [r for r in runs if any(r["run_name"].startswith(o) for o in ONLY)]
    return runs


def check_data(r):
    """Tra ve chuoi loi, hoac None neu du lieu du."""
    ds, nc = r["dataset"], N_CLASSES[r["dataset"]]
    if r["variant"] == "real" or r["pooled"]:
        fs = f"{DATA}/{ds}/real_train_fewshot/seed0"
        if not os.path.isdir(fs):
            return f"thieu tap few-shot {fs}"
        classes = [c for c in os.listdir(fs) if os.path.isdir(f"{fs}/{c}")]
        if len(classes) != nc:
            return f"{fs}: {len(classes)} lop, can {nc}"
    if r["variant"] != "real":
        d = f"{SYN}/{ds}/sd2.1/gs2.0_nis50/{SYN_SUB[r['variant']]}/train"
        if not os.path.isdir(d):
            return f"thieu anh sinh {d}"
        classes = [c for c in os.listdir(d) if os.path.isdir(f"{d}/{c}")]
        short = [c for c in classes if len(os.listdir(f"{d}/{c}")) < N_IMG]
        if len(classes) != nc or short:
            return f"{d}: {len(classes)} lop (can {nc}), lop thieu anh: {short[:3]}"
    return None


RUNS = make_runs()
for r in RUNS:
    err = check_data(r)
    r["data_error"] = err
    print("%-36s %s" % (r["run_name"], "ok" if err is None else "LOI: " + err))
print(len(RUNS), "lan chay,", sum(r["data_error"] is None for r in RUNS), "du du lieu")

In [ ]:
# ===== 13. Ham chay mot lan + doc ket qua + ghi CSV =====
import csv, datetime, glob, os, re, shutil, subprocess, time

CLF = f"{REPO}/classify"
with open(f"{REPO}/results/results_template.csv", encoding="utf-8-sig") as f:
    HEADER = next(csv.reader(f))
CODE_COMMIT = subprocess.run(["git", "-C", REPO, "rev-parse", "--short=7", "HEAD"],
                             capture_output=True, text=True).stdout.strip()
# MetricLogger in "top1: <median> (<global_avg>)"; lay global_avg = top1 cua ca tap test
TOP1 = re.compile(r"Averaged test stats:.*?top1: [-\d.]+ \(([-\d.]+)\)")


def hp_for(r):
    hp = dict(HP[r["dataset"]])
    if r["variant"] == "real" and HP_REAL_ONLY:
        hp.update(HP_REAL_ONLY)
    return hp


def flags(r, out_dir):
    hp = hp_for(r)
    f = ["--model_type=clip", f"--dataset={r['dataset']}", "--log=tensorboard", f"--output_dir={out_dir}",
         "--fewshot_seed=seed0", f"--batch_size_eval={EVAL_BS}"]
    f += [f"--{k}={v}" for k, v in hp.items()]
    if r["variant"] == "real":
        return f + ["--is_synth_train=False", "--n_shot=16"]
    f += ["--is_synth_train=True", "--sd_version=sd2.1", f"--n_img_per_cls={r['n_img']}", f"--n_shot={r['K']}"]
    if r["variant"] in ("dset", "cls"):
        f.append(f"--is_dataset_wise={r['variant'] == 'dset'}")
    if r["variant"] == "issynth":
        f.append(f"--issynth_strength={float(ISSYNTH_STRENGTH)}")
    if r["pooled"]:
        f += ["--is_pooled_fewshot=True", "--lambda_1=0.8"]
    return f


def done_runs():
    if not os.path.exists(RESULTS_CSV):
        return set()
    with open(RESULTS_CSV, encoding="utf-8") as f:
        return {row["run_name"] for row in csv.DictReader(f) if row.get("acc")}


def find_log(run_dir):
    logs = glob.glob(f"{run_dir}/**/log.log", recursive=True)
    return logs[0] if logs else None


def echo_new(run_dir, n_seen):
    log = find_log(run_dir)
    if not log:
        return n_seen
    with open(log, errors="replace") as f:
        lines = f.read().splitlines()
    for ln in lines[n_seen:]:
        if ln.startswith("Averaged test stats") or "Traceback" in ln or "Error" in ln:
            print("   ", ln[:160])
    return len(lines)


def run_one(r):
    name, hp = r["run_name"], hp_for(r)
    run_dir = f"{OUT}/{name}"
    if os.path.exists(run_dir):
        shutil.rmtree(run_dir)  # lan chay do truoc do: log.log mo che do 'a' se lan so cu
    os.makedirs(run_dir)
    cmd = ["python", "main.py"] + flags(r, run_dir)
    print(f"\n>>> {name}\n    " + " ".join(cmd[2:]))
    t0 = time.time()
    with open(f"{run_dir}/stdout.txt", "w") as out:
        p = subprocess.Popen(cmd, cwd=CLF, stdout=out, stderr=subprocess.STDOUT)
        seen = 0
        while p.poll() is None:
            time.sleep(30)
            seen = echo_new(run_dir, seen)
        echo_new(run_dir, seen)
    minutes = (time.time() - t0) / 60

    log = find_log(run_dir)
    text = open(log, errors="replace").read() if log else ""
    vals = [float(v) for v in TOP1.findall(text)]
    if "Traceback" in text or len(vals) != hp["epochs"]:
        print(f"!!! {name} HONG: {len(vals)}/{hp['epochs']} dong test, exit={p.returncode}. Cuoi log / stdout:")
        print(text[-3000:] if text else open(f"{run_dir}/stdout.txt", errors="replace").read()[-3000:])
        return None

    last, best = vals[-1], max(vals)
    hp_txt = " ".join(f"{k}={v}" for k, v in hp.items())
    note = (f"epoch cuoi; best-tren-test {best:.2f}@ep{vals.index(best)}; {hp_txt}; eval_bs={EVAL_BS}; "
            f"{minutes:.0f} phut T4")
    if r["pooled"]:
        note = "that16+sinh lambda_1=0.8; " + note
    row = {"run_name": name, "dataset": r["dataset"], "variant": r["variant"], "K": r["K"],
           "n_img": r["n_img"], "seed": "" if (r["variant"] == "sdbase" and not r["pooled"]) else 0,
           "setting": "clip-lora-pooled" if r["pooled"] else "clip-lora", "acc": f"{last:.2f}",
           "date": datetime.date.today().isoformat(), "commit": CODE_COMMIT, "note": note}
    os.makedirs(DRIVE_OUT, exist_ok=True)
    new = not os.path.exists(RESULTS_CSV)
    with open(RESULTS_CSV, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=HEADER, lineterminator="\n")
        if new:
            w.writeheader()
        w.writerow(row)
    print(f"=== {name}: top1 epoch cuoi {last:.2f} (best {best:.2f}), {minutes:.0f} phut -> {RESULTS_CSV}")

    # log len Drive (khong kem checkpoint); loi Drive se hien ra, khong bi giau
    if not KEEP_CKPT:
        for ck in glob.glob(f"{run_dir}/**/*.pth", recursive=True):
            os.remove(ck)
    shutil.copytree(run_dir, f"{DRIVE_OUT}/runs/{name}", dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns("*.pth"))
    return last

print("code commit", CODE_COMMIT, "| CSV", RESULTS_CSV)

In [ ]:
# ===== 14. Chay ca loat (bo qua lan da co trong CSV tren Drive) =====
done = done_runs()
todo = [r for r in RUNS if r["run_name"] not in done and r["data_error"] is None]
print("da xong:", sorted(done & {r["run_name"] for r in RUNS}))
print("se chay:", [r["run_name"] for r in todo])
skipped = [r["run_name"] for r in RUNS if r["data_error"] is not None]
if skipped:
    print("BO QUA (thieu du lieu, xem o 12):", skipped)

failed = []
for r in todo:
    if run_one(r) is None:
        failed.append(r["run_name"])
print("\nXONG. Hong:", failed or "khong co")

In [ ]:
# ===== 15. Bang so voi Table 1 + luu =====
%cd {REPO}
!python scripts/make_table_main.py --csv "{RESULTS_CSV}"
!python scripts/make_table_main.py --csv "{RESULTS_CSV}" --latex --out "{DRIVE_OUT}/table_main.tex"

# Viec con lai (tren MAY CUA BAN, khong lam tren Colab):
#  1. Tai {DRIVE_OUT}/clf_main.csv ve, chep de len results/clf_main.csv (file nay chi TV4 ghi).
#  2. git add results/clf_main.csv && git commit -m "[results] bang chinh ..." tren nhanh cua ban, mo PR.
# Truoc khi luu notebook: Edit > Clear all outputs, roi File > Save a copy in GitHub vao NHANH CUA BAN.